# Sphere Octant Division (`N^2`)

Divide the first sphere octant (`x, y, z >= 0`) into `N²` spherical triangles and visualize them.

- The mesh is built from a simplex lattice and normalized onto the sphere.
- Visualization uses `matplotlib` 3D plotting.
- The last cell checks side-length patterns.


## `lattice_to_octant_point`

For lattice index $(i,j,k)$ with $i+j+k=N$, define:

$$\begin{aligned}
r_{ij}&=i+j,  r_{jk}=j+k,  r_{ki}=k+i\\
\theta_{ab}&=\frac{\pi \cdot r_{ab}}{2 N}\quad (ab\in\{ij,jk,ki\})\\
\phi_{ab}&=\begin{cases}0&(r_{ab}=0)\\ \frac{\pi \cdot b}{2 r_{ab}}&(r_{ab}>0)\end{cases}
\end{aligned}$$

Then the output point is:

$$\mathbf{p}(i,j,k)=\frac{1}{3}\begin{bmatrix}
\sin\theta_{ij}\cos\phi_{ij}+\sin\theta_{ki}\sin\phi_{ki}+\cos\theta_{jk}\\
\sin\theta_{jk}\cos\phi_{jk}+\sin\theta_{ij}\sin\phi_{ij}+\cos\theta_{ki}\\
\sin\theta_{ki}\cos\phi_{ki}+\sin\theta_{jk}\sin\phi_{jk}+\cos\theta_{ij}
\end{bmatrix}. $$


In [ ]:
import numpy as np


def lattice_to_octant_point(i, j, k, N):
    ring_ij, ring_jk, ring_ki = i + j, j + k, k + i

    theta_ij = (np.pi * ring_ij) / (2.0 * N)
    theta_jk = (np.pi * ring_jk) / (2.0 * N)
    theta_ki = (np.pi * ring_ki) / (2.0 * N)
    
    phi_ij = 0.0 if ring_ij == 0 else (np.pi * j) / (2.0 * ring_ij)
    phi_jk = 0.0 if ring_jk == 0 else (np.pi * k) / (2.0 * ring_jk)
    phi_ki = 0.0 if ring_ki == 0 else (np.pi * i) / (2.0 * ring_ki)

    return np.array([
        np.sin(theta_ij) * np.cos(phi_ij) + np.sin(theta_ki) * np.sin(phi_ki) + np.cos(theta_jk),
        np.sin(theta_jk) * np.cos(phi_jk) + np.sin(theta_ij) * np.sin(phi_ij) + np.cos(theta_ki),
        np.sin(theta_ki) * np.cos(phi_ki) + np.sin(theta_jk) * np.sin(phi_jk) + np.cos(theta_ij),
    ], dtype=float) / 3


def build_octant_points(N):
    if N < 1:
        raise ValueError('N must be >= 1.')

    points = {}
    for i in range(N + 1):
        for j in range(N + 1 - i):
            k = N - i - j
            points[(i, j, k)] = lattice_to_octant_point(i, j, k, N)

    return points


def build_octant_triangle_keys(N):
    if N < 1:
        raise ValueError('N must be >= 1.')

    triangles = []
    for i in range(N):
        for j in range(N - i):
            k = N - i - j
            a = (i, j, k)
            b = (i + 1, j, k - 1)
            c = (i, j + 1, k - 1)
            triangles.append((a, b, c))

            if k >= 2:
                d = (i + 1, j + 1, k - 2)
                triangles.append((b, d, c))
    return triangles


def build_octant_mesh(N):
    points = build_octant_points(N)
    triangle_keys = build_octant_triangle_keys(N)
    tri_xyz = [np.array([points[idx] for idx in tri]) for tri in triangle_keys]
    return points, triangle_keys, tri_xyz


def build_point_index(points):
    point_keys = sorted(points.keys(), key=lambda t: (t[0] + t[1], t[0], t[1], t[2]))
    point_index = {key: idx for idx, key in enumerate(point_keys)}
    return point_keys, point_index


def triangle_side_lengths(tri):
    def ang(u, v):
        return np.arccos(np.clip(np.dot(u, v), -1.0, 1.0))

    a, b, c = tri
    return np.array([ang(a, b), ang(b, c), ang(c, a)])


## visualization

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from sphere_geometry_util import geodesic_arc


def plot_octant_division(N=6, sphere_alpha=0.18, edge_lw=0.8, figsize=(8, 8)):
    _, _, tris = build_octant_mesh(N)

    fig = plt.figure(figsize=figsize)
    ax = fig.add_subplot(111, projection='3d')

    # Octant surface patch
    th = np.linspace(0.0, np.pi / 2.0, 60)
    ph = np.linspace(0.0, np.pi / 2.0, 60)
    TH, PH = np.meshgrid(th, ph)
    X = np.sin(TH) * np.cos(PH)
    Y = np.sin(TH) * np.sin(PH)
    Z = np.cos(TH)
    ax.plot_surface(X, Y, Z, color='lightsteelblue', alpha=sphere_alpha, linewidth=0, antialiased=True)

    # Triangle faces (semi-transparent for readability)
    poly = Poly3DCollection([tri for tri in tris], facecolors='cornflowerblue', edgecolors='none', alpha=0.25)
    ax.add_collection3d(poly)

    # Geodesic edges on the sphere
    for tri in tris:
        for e0, e1 in ((0, 1), (1, 2), (2, 0)):
            arc = geodesic_arc(tri[e0], tri[e1], samples=20)
            ax.plot(arc[:, 0], arc[:, 1], arc[:, 2], color='k', linewidth=edge_lw, alpha=0.75)

    # Coordinate axes
    ax.plot([0, 1.1], [0, 0], [0, 0], color='r', linewidth=1.2)
    ax.plot([0, 0], [0, 1.1], [0, 0], color='g', linewidth=1.2)
    ax.plot([0, 0], [0, 0], [0, 1.1], color='b', linewidth=1.2)

    ax.text(1.12, 0, 0, 'x')
    ax.text(0, 1.12, 0, 'y')
    ax.text(0, 0, 1.12, 'z')

    ax.set_xlim(0, 1.05)
    ax.set_ylim(0, 1.05)
    ax.set_zlim(0, 1.05)
    ax.set_box_aspect((1, 1, 1))
    ax.view_init(elev=28, azim=38)
    ax.set_title(f'Octant spherical-triangle division: N={N}, count={len(tris)}')

    plt.tight_layout()
    plt.show()

    return tris

In [ ]:
# Example run
N = 16
tris = plot_octant_division(N=N)
print(f'number of spherical triangles = {len(tris)} (expected: {N**2})')

In [ ]:
# List all coordinate points (with indices) and all triangle vertex indices

points, triangle_keys, _ = build_octant_mesh(N)
point_keys, point_index = build_point_index(points)

print('=== Points (index: lattice_index -> [x, y, z]) ===')
for key in point_keys:
    idx = point_index[key]
    x, y, z = points[key]
    print(f'{idx:3d}: {key} -> [{x:.8f}, {y:.8f}, {z:.8f}]')

print('\n=== Triangles (triangle_id: point_index_triplet) ===')
for t_id, tri in enumerate(triangle_keys):
    ids = tuple(point_index[v] for v in tri)
    print(f'{t_id:3d}: {ids}')

print(f'\npoint_count = {len(point_keys)}')
print(f'triangle_count = {len(triangle_keys)} (expected: {N**2})')

# Verify that all triangle normals point outward
inward_ids = []
for t_id, tri in enumerate(triangle_keys):
    i, j, k = [points[v] for v in tri]
    normal = np.cross(j - i, k - i)
    centroid = (i + j + k) / 3.0
    if np.dot(normal, centroid) <= 0.0:
        inward_ids.append(t_id)

all_outward = len(inward_ids) == 0
print(f'\nall_outward_normals = {all_outward}')
if not all_outward:
    print('inward_triangle_ids =', inward_ids)


# Permutation-equivariance test: f(perm(i,j,k)) == perm(f(i,j,k))
import itertools

perms = list(itertools.permutations([0, 1, 2]))
max_perm_err = 0.0
worst_case = None

for i in range(N + 1):
    for j in range(N + 1 - i):
        k = N - i - j
        v_base = lattice_to_octant_point(i, j, k, N)
        base_idx = np.array([i, j, k], dtype=int)

        for p in perms:
            ip = base_idx[list(p)]
            v_perm_input = lattice_to_octant_point(int(ip[0]), int(ip[1]), int(ip[2]), N)
            v_perm_output = v_base[list(p)]
            err = float(np.linalg.norm(v_perm_input - v_perm_output))

            if err > max_perm_err:
                max_perm_err = err
                worst_case = ((i, j, k), p)

print(f'max_permutation_error = {max_perm_err:.3e}')
print('worst_case =', worst_case)


In [ ]:
# Planar triangle area distribution

points, triangle_keys, _ = build_octant_mesh(N)

areas = []
for tri in triangle_keys:
    a, b, c = [points[v] for v in tri]
    area = 0.5 * np.linalg.norm(np.cross(b - a, c - a))
    areas.append(area)

areas = np.array(areas, dtype=float)

print('=== Planar Triangle Area Distribution ===')
print(f'triangle_count = {areas.size}')
print(f'min   = {areas.min():.10f}')
print(f'max   = {areas.max():.10f}')
print(f'mean  = {areas.mean():.10f}')
print(f'median= {np.median(areas):.10f}')
print(f'std   = {areas.std(ddof=0):.10f}')

q = np.quantile(areas, [0.0, 0.25, 0.5, 0.75, 1.0])
print('quantiles (0,25,50,75,100)% =', [f'{v:.10f}' for v in q])

bins = min(20, max(5, int(np.sqrt(areas.size))))
hist, edges = np.histogram(areas, bins=bins)

# Visualization with matplotlib
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(areas, bins=bins, color='steelblue', edgecolor='black', alpha=0.85)
ax.axvline(areas.mean(), color='crimson', linestyle='--', linewidth=1.5, label=f'mean={areas.mean():.6f}')
ax.axvline(np.median(areas), color='darkgreen', linestyle=':', linewidth=1.5, label=f'median={np.median(areas):.6f}')
ax.set_title(f'Planar Triangle Area Distribution (N={N}, count={areas.size})')
ax.set_xlabel('Triangle area')
ax.set_ylabel('Count')
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()
